# Analyse ferroviaire avec Spark — rendu d’origine

Code et sorties reconstruits depuis l’export HTML Databricks fourni. Le code n’a pas été modifié et les résultats n’ont pas été réexécutés lors de cette conversion. Les numéros d’exécution correspondent à l’ordre des cellules dans l’export. Voir [les adaptations](../docs/adaptations.md) pour la version portfolio.

In [1]:
##  PARTIE A  ##
# Exercice 1 - Chargement
trains = spark.read.csv ("/Volumes/workspace/default/tptrains/trains.csv", header=True, inferSchema=True)
# Afficher les 10 premières lignes
print("10 premières lignes :")
trains.limit(10).show()
# Afficher le schéma
print("Schéma du DataFrame:")
trains.printSchema()
# Afficher le nombre total de trajets
print(f"Nombre total de trajets : {trains.count()}")

10 premières lignes :
+-------+----------+-----------------+---------------+-----------+----------+-----------+------------+----------+---------+---------+
|trip_id|      date|departure_station|arrival_station|    company|train_type|distance_km|duration_min|passengers|delay_min|cancelled|
+-------+----------+-----------------+---------------+-----------+----------+-----------+------------+----------+---------+---------+
|  10001|2026-11-04|            Paris|           Lyon|       SNCF|       TGV|        423|         153|       272|       30|       No|
|  10002|2026-08-14|            Paris|      Marseille|       SNCF|       TGV|        668|         241|       471|       25|       No|
|  10003|2026-09-09|            Paris|          Lille|       SNCF|Intercités|        185|         110|       350|        5|       No|
|  10004|2026-09-20|            Paris|       Bordeaux|       SNCF|Intercités|        515|         316|       302|       27|       No|
|  10005|2026-06-08|            Paris|  

In [2]:
# Exercice 2 - Sélection
# Afficher uniquement les colonnes departure_station/arrival_station/distance_km/duration_min/delay_min
# Afficher les 20 premières lignes du résultat
print("20 premières lignes de la sélection :")
trains.select("departure_station", "arrival_station", "distance_km", "duration_min", "delay_min").limit(20).show()


20 premières lignes de la sélection :
+-----------------+---------------+-----------+------------+---------+
|departure_station|arrival_station|distance_km|duration_min|delay_min|
+-----------------+---------------+-----------+------------+---------+
|            Paris|           Lyon|        423|         153|       30|
|            Paris|      Marseille|        668|         241|       25|
|            Paris|          Lille|        185|         110|        5|
|            Paris|       Bordeaux|        515|         316|       27|
|            Paris|       Toulouse|        591|         200|       15|
|            Paris|         Nantes|        405|         134|       24|
|            Paris|     Strasbourg|        450|         270|        4|
|            Paris|         Rennes|        380|         212|        8|
|            Paris|    Montpellier|        610|         510|       13|
|             Lyon|          Paris|        423|         244|        3|
|             Lyon|      Marseille|    

In [3]:
# Exercice 3 - Valeurs distinctes
print("---Gares de départ---")
trains.select("departure_station").distinct().orderBy("departure_station").show()
print("---Compagnies---")
trains.select("company").distinct().orderBy("company").show()
print("---Types de trains---")
trains.select("train_type").distinct().orderBy("train_type").show()

---Gares de départ---
+-----------------+
|departure_station|
+-----------------+
|         Bordeaux|
|            Lille|
|             Lyon|
|        Marseille|
|      Montpellier|
|           Nantes|
|            Paris|
|           Rennes|
|       Strasbourg|
|         Toulouse|
+-----------------+

---Compagnies---
+-----------+
|    company|
+-----------+
|   HexaRail|
|RailExpress|
|       SNCF|
+-----------+

---Types de trains---
+----------+
|train_type|
+----------+
|Intercités|
|       TER|
|       TGV|
+----------+



In [4]:
# Exercice 4 - Filtrage
# Afficher uniquement les colonnes trip_id / departure_station / arrival_station / train_type / distance_km
# Filtre sur les trajets départ Paris, distance > 300km, train non annulé
print("Filtre sur les trajets départ Paris, distance > 300km, train non annulé : ")
trains.select(
    "trip_id",
    "departure_station",
    "arrival_station", "train_type",
    "distance_km"
    ).filter(
        (trains.departure_station == "Paris") &
        (trains.distance_km > 300) &
        (trains.cancelled == "No")
    ).show()


Filtre sur les trajets départ Paris, distance > 300km, train non annulé : 
+-------+-----------------+---------------+----------+-----------+
|trip_id|departure_station|arrival_station|train_type|distance_km|
+-------+-----------------+---------------+----------+-----------+
|  10001|            Paris|           Lyon|       TGV|        423|
|  10002|            Paris|      Marseille|       TGV|        668|
|  10004|            Paris|       Bordeaux|Intercités|        515|
|  10005|            Paris|       Toulouse|       TGV|        591|
|  10006|            Paris|         Nantes|       TGV|        405|
|  10007|            Paris|     Strasbourg|Intercités|        450|
|  10008|            Paris|         Rennes|Intercités|        380|
|  10009|            Paris|    Montpellier|       TER|        610|
|  10091|            Paris|           Lyon|       TGV|        423|
|  10092|            Paris|      Marseille|       TGV|        668|
|  10094|            Paris|       Bordeaux|       TGV|

In [5]:
##  PARTIE B  ##
# Création de la vue temporaire SQL
trains.createOrReplaceTempView("trains")
# Exercice 5 - Statistiques générales
print("Statistiques générales : ")
spark.sql("""
    SELECT
        MIN(distance_km) AS minDistance,
        MAX(distance_km) AS maxDistance,
        ROUND(AVG(distance_km), 2) AS avgDistance,
        ROUND(AVG(duration_min), 2) AS avgDuration,
        ROUND(AVG(delay_min), 2) AS avgDelay
    FROM trains
    WHERE cancelled = 'No'
""").show()       

Statistiques générales : 
+-----------+-----------+-----------+-----------+--------+
|minDistance|maxDistance|avgDistance|avgDuration|avgDelay|
+-----------+-----------+-----------+-----------+--------+
|        185|        887|     553.82|     246.25|   14.84|
+-----------+-----------+-----------+-----------+--------+



In [6]:
# Exercice 6 - Retards les plus importants
# Afficher uniquement les colonnes trip_id / departure_station / arrival_station / train_type / delay_min
# Afficher les 10 trajets ayant subi les retards les plus importants, du retard le plus important au moins important
print("TOP10 des retards les plus importants : ")
spark.sql("""
    SELECT
        trip_id,
        departure_station,
        arrival_station,
        train_type,
        delay_min
    FROM trains
    ORDER BY delay_min DESC
    LIMIT 10
""").show()

TOP10 des retards les plus importants : 
+-------+-----------------+---------------+----------+---------+
|trip_id|departure_station|arrival_station|train_type|delay_min|
+-------+-----------------+---------------+----------+---------+
|  14976|        Marseille|         Rennes|       TGV|      110|
|  10201|        Marseille|          Lille|       TGV|      110|
|  15244|        Marseille|         Nantes|       TGV|      107|
|  14252|        Marseille|       Bordeaux|Intercités|      100|
|  14656|       Strasbourg|      Marseille|       TGV|      100|
|  13801|        Marseille|          Lille|Intercités|      100|
|  11236|       Strasbourg|      Marseille|       TGV|       93|
|  12185|        Marseille|     Strasbourg|Intercités|       93|
|  11196|        Marseille|         Rennes|       TGV|       91|
|  14842|       Strasbourg|    Montpellier|       TGV|       91|
+-------+-----------------+---------------+----------+---------+



In [7]:
# Exercice 7 - Annulations
# Calcul du nombre de trajets annulés/non annulés, à présenter sous forme de tableau
spark.sql("SELECT DISTINCT cancelled FROM trains").show()
print ("Nombre de trajets annulés/non annulés :")
cancellation = spark.sql("""
    SELECT
        cancelled,
        COUNT(*) AS numberTrips
    FROM trains
    GROUP BY cancelled
""")
display(cancellation)


+---------+
|cancelled|
+---------+
|      Yes|
|       No|
+---------+

Nombre de trajets annulés/non annulés :


cancelled,numberTrips
Yes,257
No,6943


In [8]:
# Exercice 8 - Classification des retards
# Ne pas prendre en compte les trajets annulés
# Création d'une nouvelle colonne delayLevel pour catégoriser les retards
#   - 0 à 5 minutes : On time
#   - 5+ à 15 minutes : Small delay
#   - 15+ à 30 minutes : Medium delay
#   - >30 minutes : Large dealy
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW trains_delay_category AS
    SELECT *,
        CASE
            WHEN delay_min <= 5 THEN 'On time'
            WHEN delay_min > 5 AND delay_min <= 15 THEN 'Small delay'
            WHEN delay_min > 15 AND delay_min <= 30 THEN 'Medium delay'
            WHEN delay_min > 30 THEN 'Large delay'
        END AS delayLevel
    FROM trains
    WHERE cancelled = 'No'
""")
# Calcul du nombre de trajets de chaque categorie
print("Nombre de trajets pour chaque catégorie de retard :")
spark.sql("""
    SELECT
        delayLevel,
        COUNT(*) AS numberTrips
        FROM trains_delay_category
        GROUP BY delayLevel
        ORDER BY CASE delayLevel
            WHEN 'On time' THEN 1
            WHEN 'Small delay' THEN 2
            WHEN 'Medium delay' THEN 3
            WHEN 'Large delay' THEN 4
        END
""").show()


Nombre de trajets pour chaque catégorie de retard :
+------------+-----------+
|  delayLevel|numberTrips|
+------------+-----------+
|     On time|       2934|
| Small delay|       1829|
|Medium delay|       1424|
| Large delay|        756|
+------------+-----------+



In [9]:
################
##  PARTIE C  ##
################
# Exercice 9
# Les trajets annulés ne sont pas pris en compte
# Pour chaque gare de départ, afficher dans un tableau : nb de trajet décroissant, nb total de passagers, distance moy, retard moy.
stations =spark.sql("""
    SELECT
        departure_station,
        COUNT(*) AS numberTrips,
        SUM(passengers) AS totalPassengers,
        ROUND(AVG(distance_km), 2) AS avgDistance,
        ROUND(AVG(delay_min), 2) AS avgDelay
    FROM trains
    WHERE cancelled = 'No'
    GROUP BY departure_station
    ORDER BY numberTrips DESC, departure_station
""")
display(stations)



departure_station,numberTrips,totalPassengers,avgDistance,avgDelay
Lyon,702,244684,502.1,13.78
Nantes,701,234468,529.83,13.52
Paris,697,232984,470.48,14.14
Strasbourg,697,246226,657.03,17.36
Bordeaux,694,240653,546.34,14.62
Lille,694,241476,541.06,14.73
Marseille,690,230612,613.45,18.23
Montpellier,690,228153,555.1,14.21
Toulouse,690,241388,555.24,14.24
Rennes,688,225880,568.85,13.57


In [10]:
# Exercice 10 - Fréquentation des gares
# Les trajets annulés ne sont pas pris en compte (voir la variable 'stations' de l'exercice 9)
stations.createOrReplaceTempView("stations")
# 1. Top5 des gares de départ ayant transportés le plus de passagers
print("Top5 des gares de départ ayant transportés le plus de passagers :")
spark.sql("""
            SELECT
                departure_station,
                totalPassengers,
                avgDelay
            FROM stations
            ORDER BY totalPassengers DESC
            LIMIT 5
""").show()
# 2. Top5 des gares de départ ayant enregistré le plus de trajets
print("Top5 des gares de départ ayant enregistré le plus de trajets :")
spark.sql("""
    SELECT
        departure_station,
        numberTrips,
        totalPassengers
    FROM stations
    ORDER BY numberTrips DESC
    LIMIT 5
""").show()
# 3. Top5 des gares ayant le retard moyen le plus élevé
print("Top5 des gares ayant le retard moyen le plus élevé :")
spark.sql("""
    SELECT
        departure_station,
        avgDelay,
        totalPassengers
    FROM stations
    ORDER BY avgDelay DESC
    LIMIT 5
""").show()

Top5 des gares de départ ayant transportés le plus de passagers :
+-----------------+---------------+--------+
|departure_station|totalPassengers|avgDelay|
+-----------------+---------------+--------+
|       Strasbourg|         246226|   17.36|
|             Lyon|         244684|   13.78|
|            Lille|         241476|   14.73|
|         Toulouse|         241388|   14.24|
|         Bordeaux|         240653|   14.62|
+-----------------+---------------+--------+

Top5 des gares de départ ayant enregistré le plus de trajets :
+-----------------+-----------+---------------+
|departure_station|numberTrips|totalPassengers|
+-----------------+-----------+---------------+
|             Lyon|        702|         244684|
|           Nantes|        701|         234468|
|            Paris|        697|         232984|
|       Strasbourg|        697|         246226|
|         Bordeaux|        694|         240653|
+-----------------+-----------+---------------+

Top5 des gares ayant le retard m

In [11]:
# Exercice 11 - Interprétation
# ●	Les gares ayant le plus grand nombre de trajets sont-elles également celles transportant le plus de passagers ?
# ●	Les gares les plus fréquentées sont-elles celles qui présentent les retards moyens les plus élevés ?
print("""Les classement de gares ne sont pas identiques : \n
Les villes assurant le plus de trajets ne sont pas les plus fréquentées \n
(ex Paris : 697 trajets pour seulement 232984 passagers vs Bordeaux : 694 trajets pour 240653 passagers). \n
Les villes les plus fréquentées ne sont pas celles présentant les retards moyens les plus élevés \n
(ex Lyon est dans le TOP5 gare les plus fréquentées mais n'est pas dans le TOP5 des gares présentant les retards moyens les plus élevés) \n
(ex Marseille est dans le TOP5 des gares présentant les retards moyens les plus élevés mais n'est pas dans le TOP5 des gares les plus fréquentées)
""")

Les classement de gares ne sont pas identiques : 

Les villes assurant le plus de trajets ne sont pas les plus fréquentées 

(ex Paris : 697 trajets pour seulement 232984 passagers vs Bordeaux : 694 trajets pour 240653 passagers). 

Les villes les plus fréquentées ne sont pas celles présentant les retards moyens les plus élevés 

(ex Lyon est dans le TOP5 gare les plus fréquentées mais n'est pas dans le TOP5 des gares présentant les retards moyens les plus élevés) 

(ex Marseille est dans le TOP5 des gares présentant les retards moyens les plus élevés mais n'est pas dans le TOP5 des gares les plus fréquentées)



In [12]:
################
##  PARTIE D  ##
################
# Exercice 12 - Statistiques par ligne
# Les trajets annulés ne sont pas pris en compte
# Pour chaque ligne ferroviaire, afficher : nb de trajet, nb total de passagers, retard moy, retard maximum.
print("Statistiques par ligne ferroviaire :")
lines = spark.sql("""
    SELECT
        departure_station,
        arrival_station,
        COUNT(*) AS numberTrips,
        SUM(passengers) AS totalPassengers,
        ROUND(AVG(delay_min), 2) AS avgDelay,
        MAX(delay_min) AS maxDelay
    FROM trains
    WHERE cancelled = 'No'
    GROUP BY departure_station, arrival_station
""")
display(lines)
lines.createOrReplaceTempView("lines")



Statistiques par ligne ferroviaire :


departure_station,arrival_station,numberTrips,totalPassengers,avgDelay,maxDelay
Paris,Lille,79,20462,13.97,86
Lyon,Paris,78,27156,14.44,80
Marseille,Rennes,75,26874,23.19,110
Lille,Bordeaux,73,25946,13.51,85
Bordeaux,Lyon,76,26810,13.5,82
Toulouse,Montpellier,75,27608,16.84,80
Strasbourg,Lyon,75,24871,17.11,86
Rennes,Marseille,76,24613,13.68,78
Rennes,Bordeaux,76,26803,12.41,81
Lyon,Lille,75,25524,13.57,85


In [13]:
# Exercice 13 - Comparaison des lignes
# Classement A :Top10 des lignes ayant le plus grand nombre de passagers
print("Top10 des lignes ayant le plus grand nombre de passagers :")
spark.sql("""
    SELECT
        departure_station,
        arrival_station,
        totalPassengers,
        numberTrips,
        avgDelay
    FROM lines
    ORDER BY totalPassengers DESC
    LIMIT 10
""").show()

# Interprétation
print("Les lignes les plus fréquentées sont celles présentant le plus de trajets (sauf Strasbourg-Paris).")
print ("Sur ces lignes fréquentées, les retards importants sont sur Lille-Starsbourg et Lille-Montpellier. \n \n")


# Classement B :Top10 des lignes ayant le retard moyen le plus élevé
print("Top10 des lignes ayant le retard moyen le plus élevé :")
spark.sql("""
    SELECT
        departure_station,
        arrival_station,
        avgDelay,
        maxDelay,
        numberTrips,
        totalPassengers
    FROM lines
    ORDER BY avgDelay DESC
    LIMIT 10
""").show()
# Interprétation
print("Les lignes ayant le retard moyen le plus élevé sont celles qui présentent un retard maximum important.")

Top10 des lignes ayant le plus grand nombre de passagers :
+-----------------+---------------+---------------+-----------+--------+
|departure_station|arrival_station|totalPassengers|numberTrips|avgDelay|
+-----------------+---------------+---------------+-----------+--------+
|             Lyon|     Strasbourg|          30115|         78|   12.17|
|            Lille|     Strasbourg|          29690|         80|    18.5|
|       Strasbourg|          Paris|          29676|         76|   13.96|
|            Lille|         Rennes|          29419|         79|   13.53|
|         Bordeaux|          Paris|          29234|         78|   16.76|
|             Lyon|       Toulouse|          29149|         79|   13.59|
|            Lille|    Montpellier|          29136|         80|    17.7|
|             Lyon|         Rennes|          28999|         79|   12.29|
|        Marseille|       Toulouse|          28963|         79|   16.54|
|           Nantes|     Strasbourg|          28858|         79|  

In [14]:
# Exercice 14 - Fiabilité de l'analyse
# Top10 des lignes ayant le retard moyen le plus élevé avec des lignes ayant réalisé au moins 30 trajets
print("Top10 des lignes ayant le retard moyen le plus élevé avec des lignes ayant réalisé au moins 30 trajets :")
spark.sql("""
    SELECT
        departure_station,
        arrival_station,
        avgDelay,
        maxDelay,
        numberTrips,
        totalPassengers
    FROM lines
    WHERE numberTrips >= 30
    ORDER BY avgDelay DESC
    LIMIT 10
""").show()

# Comparatif avec le résultat de l'exercice 13
print("""Les résultats sont identiques à l'exercice 13 car les lignes comportent toutes plus de 30 trajets. \n
L'analyse réalisée lors de l'exercice 13 est donc fiable au regard du nombre ds trajets réalisés sur les lignes.\n \n
""")



Top10 des lignes ayant le retard moyen le plus élevé avec des lignes ayant réalisé au moins 30 trajets :
+-----------------+---------------+--------+--------+-----------+---------------+
|departure_station|arrival_station|avgDelay|maxDelay|numberTrips|totalPassengers|
+-----------------+---------------+--------+--------+-----------+---------------+
|        Marseille|         Rennes|   23.19|     110|         75|          26874|
|        Marseille|          Lille|   22.28|     110|         75|          26494|
|       Strasbourg|      Marseille|   20.23|     100|         78|          28488|
|        Marseille|    Montpellier|   20.04|      90|         78|          20871|
|       Strasbourg|       Bordeaux|   19.35|      87|         78|          28317|
|         Toulouse|          Paris|   18.65|      89|         77|          27591|
|       Strasbourg|         Nantes|   18.64|      78|         78|          26439|
|            Lille|     Strasbourg|    18.5|      89|         80|          

In [15]:
################
##  PARTIE E  ##
################

# Exercice 15 - Analyse des types de trains
# Afficher pour chaque type de train : nb de trajets non annulés, distance moy, durée moy, retard moy, nb de passagers
print("Analyse par types de trains :")
trains = spark.sql("""
    SELECT
        train_type,
        COUNT(*) AS numberTrips,
        ROUND(AVG(distance_km), 2) AS avgDistance,
        ROUND(AVG(duration_min), 2) AS avgDuration,
        ROUND(AVG(delay_min), 2) AS avgDelay,
        SUM(passengers) AS totalPassengers
    FROM trains
    WHERE cancelled = 'No'
    GROUP BY train_type
""").show()

Analyse par types de trains :
+----------+-----------+-----------+-----------+--------+---------------+
|train_type|numberTrips|avgDistance|avgDuration|avgDelay|totalPassengers|
+----------+-----------+-----------+-----------+--------+---------------+
|Intercités|       1809|     538.42|     299.29|   14.44|         486172|
|       TGV|       4170|     583.32|      196.1|   14.92|        1710299|
|       TER|        964|     455.16|     363.63|   15.22|         170053|
+----------+-----------+-----------+-----------+--------+---------------+



In [16]:
# Exercice 16 - Types de trains et retards
# Les trajets annulés ne sont pas pris en compte
# Affichage des types de trains dont le retard moyen est supérieur à 10 minutes
print("Types de trains dont le retard moyen est supérieur à 10 minutes :")
spark.sql("""
    SELECT
        train_type,
        COUNT(*) AS numberTrips,
        ROUND(AVG(delay_min), 2) AS avgDelay
    FROM trains
    WHERE cancelled = 'No'
    GROUP BY train_type
    HAVING AVG(delay_min) > 10
""").show()

Types de trains dont le retard moyen est supérieur à 10 minutes :
+----------+-----------+--------+
|train_type|numberTrips|avgDelay|
+----------+-----------+--------+
|Intercités|       1809|   14.44|
|       TGV|       4170|   14.92|
|       TER|        964|   15.22|
+----------+-----------+--------+



In [17]:
# Exercice 17 - Analyse des compagnies
# Affichage pour chaque compagnie : nb de trajets décroissant, nb de trajets annulés, nb total de passagers, retard moy des trajets non annulés
print("Analyse par compagnies :")
spark.sql("""
    SELECT
        company,
        COUNT(*) AS numberTrips,
        SUM(CASE WHEN cancelled = 'Yes' THEN 1 ELSE 0 END) AS cancelledTrips,
        SUM(passengers) AS totalPassengers,
        ROUND(AVG(CASE WHEN cancelled = 'No' THEN delay_min END), 2) AS avgDelay
    FROM trains
    GROUP BY company
    ORDER BY numberTrips DESC
""").show()


Analyse par compagnies :
+-----------+-----------+--------------+---------------+--------+
|    company|numberTrips|cancelledTrips|totalPassengers|avgDelay|
+-----------+-----------+--------------+---------------+--------+
|       SNCF|       4774|           175|        1572622|   14.98|
|RailExpress|       1391|            47|         453103|   14.46|
|   HexaRail|       1035|            35|         340799|   14.69|
+-----------+-----------+--------------+---------------+--------+



In [18]:
# Exercie 18 - Compagnies et type de trains
# Affichage du TOP10 pour chaque combinaison company + train_type : nb de trajets total (annulé et non annulés), nb de passagers décroissant, retard moy (sur trajets non annulés)
print("TOP10 du plus grand nombre de passagers pour chaque combinaison company + train_type :")
spark.sql("""
    SELECT
        company,
        train_type,
        COUNT(*) AS numberTrips,
        SUM(passengers) AS totalPassengers,
        ROUND(AVG(CASE WHEN cancelled = 'No' THEN delay_min END), 2) AS avgDelay
    FROM trains
    GROUP BY company, train_type
    ORDER BY totalPassengers DESC
    LIMIT 10
""").show()


TOP10 du plus grand nombre de passagers pour chaque combinaison company + train_type :
+-----------+----------+-----------+---------------+--------+
|    company|train_type|numberTrips|totalPassengers|avgDelay|
+-----------+----------+-----------+---------------+--------+
|       SNCF|       TGV|       2872|        1138545|   15.06|
|       SNCF|Intercités|       1263|         325074|   14.33|
|RailExpress|       TGV|        826|         321564|   14.29|
|   HexaRail|       TGV|        623|         250190|   15.11|
|       SNCF|       TER|        639|         109003|   15.88|
|RailExpress|Intercités|        368|          96985|   15.21|
|   HexaRail|Intercités|        260|          64113|   13.88|
|RailExpress|       TER|        197|          34554|   13.76|
|   HexaRail|       TER|        152|          26496|   14.32|
+-----------+----------+-----------+---------------+--------+



In [19]:
################
##  PARTIE F  ##
################
# Exercice 19 - Indicateur de ponctualité
# Les trajets annulés ne sont pas pris en compte
# Affichage pour chaque gare : nb trajets non annulés, nb passagers, retard moy,retard max, nb trajets retardés > 15 minutes, taux de trajets retardés > 15 minutes
print("Indicateur de ponctualité par gare de départ")
punctuality = spark.sql("""
    SELECT
        departure_station,
        COUNT(*) AS numberTrips,
        SUM(passengers) AS totalPassengers,
        ROUND(AVG(delay_min), 2) AS avgDelay,
        MAX(delay_min) AS maxDelay,
        SUM(CASE WHEN delay_min > 15 THEN 1 ELSE 0 END) AS delayedTrips,
        ROUND(100*delayedTrips/numberTrips, 2) AS delayrate
    FROM trains
    WHERE cancelled = 'No'
    GROUP BY departure_station
    ORDER BY delayrate DESC
""")
punctuality.show()

Indicateur de ponctualité par gare de départ
+-----------------+-----------+---------------+--------+--------+------------+---------+
|departure_station|numberTrips|totalPassengers|avgDelay|maxDelay|delayedTrips|delayrate|
+-----------------+-----------+---------------+--------+--------+------------+---------+
|        Marseille|        690|         230612|   18.23|     110|         291|    42.17|
|       Strasbourg|        697|         246226|   17.36|     100|         283|     40.6|
|      Montpellier|        690|         228153|   14.21|      90|         212|    30.72|
|         Bordeaux|        694|         240653|   14.62|      90|         206|    29.68|
|         Toulouse|        690|         241388|   14.24|      90|         204|    29.57|
|             Lyon|        702|         244684|   13.78|      90|         205|     29.2|
|            Lille|        694|         241476|   14.73|      90|         202|    29.11|
|            Paris|        697|         232984|   14.14|      89|

In [20]:
# Exercice 20 - Rapport sur les lignes
# Les trajets annulés ne sont pas pris en compte, les lignes possédant moins de 30 trajets sont exclues
# Affichage d'un rapport pour chaque ligne ferroviaire : nb trajets non annulés, nb passagers, nb moy de passagers, durée moy, retard moy,retard max
print("Rapport pour chaque ligne ferroviaire :")
line_report = spark.sql("""
    SELECT
        departure_station,
        arrival_station,
        COUNT(*) AS numberTrips,
        SUM(passengers) AS totalPassengers,
        ROUND(AVG(passengers), 2) AS avgPassengers,
        ROUND(AVG(duration_min), 2) AS avgDuration,
        ROUND(AVG(delay_min), 2) AS avgDelay,
        MAX(delay_min) AS maxDelay
    FROM trains
    WHERE cancelled = 'No'
    GROUP BY departure_station, arrival_station
    HAVING numberTrips >= 30
    ORDER BY totalPassengers DESC
""")
display(line_report)

Rapport pour chaque ligne ferroviaire :


departure_station,arrival_station,numberTrips,totalPassengers,avgPassengers,avgDuration,avgDelay,maxDelay
Lyon,Strasbourg,78,30115,386.09,161.58,12.17,84
Lille,Strasbourg,80,29690,371.13,198.78,18.5,89
Strasbourg,Paris,76,29676,390.47,176.79,13.96,85
Lille,Rennes,79,29419,372.39,169.7,13.53,83
Bordeaux,Paris,78,29234,374.79,233.17,16.76,85
Lyon,Toulouse,79,29149,368.97,218.63,13.59,70
Lille,Montpellier,80,29136,364.2,317.1,17.7,80
Lyon,Rennes,79,28999,367.08,292.65,12.29,70
Marseille,Toulouse,79,28963,366.62,220.99,16.54,81
Nantes,Strasbourg,79,28858,365.29,330.43,9.81,73


In [21]:
# Exercice 21 - Analyse finale
print("""
    Conclusion :
    L'analyse porte sur 7200 trajets (dont 6943 non annulés) depuis 10 gares de départ en France.
    Le retard moyen est de 14.84 minutes.
    Strasbourg est la première gare transportant le plus de passagers avec 246 226 passagers.
    Lyon est deuxième avec 2444 684 passagers et présente le plus de trajets (702 trajets).
    La ligne Lyon -> Strasbourg est la plus fréquentée, avec 30 115 passagers.
    Elle devance la ligne Lille -> Strasbourg (29 690 passagers) et la ligne Strasbourg -> Paris (29 676 passagers).
    Marseille présente de loin le taux de trajets avec retard supérieur à 15 minutes le plus élevé (42.17%), suivi de Strasbourg (40.60%). Marseille et Strasbourg occupent le haut du Top10 des lignes ayant le retard moyen le plus élevé (avec des retards maximums supérieurs à 90 minutes). Ces 2 gares méritent une attention particulière.
    Le retard moyen le plus élevé par ligne concerne Marseille -> Rennes (23.19 minutes sur 75 trajets) suivi de Marseille -> Lille (22.28 minutes sur 75 trajets).
    Les lignes les plus fréquentées ne sont pas nécessairement les plus en retard : ainsi, Lyon -> Strasbourg compte 30 115 passagers mais présente un retard moyen de 12.17 minutes sur 78 trajets.
    La société devrait examiner en priorité la ponctualité au départ de Marseille et Strasbourg, puis rechercher les causes de retard sur les lignes Marseille -> Rennes et Marseille -> Lille.
""")



    Conclusion :
    L'analyse porte sur 7200 trajets (dont 6943 non annulés) depuis 10 gares de départ en France.
    Le retard moyen est de 14.84 minutes.
    Strasbourg est la première gare transportant le plus de passagers avec 246 226 passagers.
    Lyon est deuxième avec 2444 684 passagers et présente le plus de trajets (702 trajets).
    La ligne Lyon -> Strasbourg est la plus fréquentée, avec 30 115 passagers.
    Elle devance la ligne Lille -> Strasbourg (29 690 passagers) et la ligne Strasbourg -> Paris (29 676 passagers).
    Marseille présente de loin le taux de trajets avec retard supérieur à 15 minutes le plus élevé (42.17%), suivi de Strasbourg (40.60%). Marseille et Strasbourg occupent le haut du Top10 des lignes ayant le retard moyen le plus élevé (avec des retards maximums supérieurs à 90 minutes). Ces 2 gares méritent une attention particulière.
    Le retard moyen le plus élevé par ligne concerne Marseille -> Rennes (23.19 minutes sur 75 trajets) suivi de Marseille